In [ ]:
import pandas as pd

keys = ['adm1', 'province', 'adm2_code', 'year', 'month']

def load_clean_parquet(filename, value_columns):
    df = pd.read_parquet(filename)

    # Keeps only analysis columns; removes .geo, system:index, and other export metadata.
    return df[keys + value_columns].copy()


df_ndvi = load_clean_parquet(
    'province_ndvi_monthly.parquet',
    ['ndvi', 'ndvi_valid']
)

df_alan = load_clean_parquet(
    'province_alan_monthly.parquet',
    ['alan', 'alan_valid']
)

df_lst = load_clean_parquet(
    'province_lst_day_monthly.parquet',
    ['lst_day', 'lst_valid']
)

df_rain = load_clean_parquet(
    'province_rain_monthly.parquet',
    ['rain_mm', 'rain_valid']
)

province_monthly = (
    df_ndvi
    .merge(df_alan, on=keys, how='outer', validate='one_to_one')
    .merge(df_lst, on=keys, how='outer', validate='one_to_one')
    .merge(df_rain, on=keys, how='outer', validate='one_to_one')
    .sort_values(['adm2_code', 'year', 'month'])
    .reset_index(drop=True)
)

print('Rows:', len(province_monthly))
print('Expected:', province_monthly['adm2_code'].nunique() * 144)

print('\nMissing values:')
print(province_monthly[
    ['alan', 'ndvi', 'lst_day', 'rain_mm']
].isna().sum())

province_monthly.to_parquet(
    'province_monthly.parquet',
    index=False
)

print('✅ Saved clean merged file: province_monthly.parquet')

Rows: 12240
Expected: 12240

Missing values:
alan        10
ndvi       320
lst_day     36
rain_mm      0
dtype: int64
✅ Saved clean merged file: province_monthly.parquet


In [12]:
from google.colab import drive
from pathlib import Path

import numpy as np
import pandas as pd


# -------------------------------------------------------------------
# 1. Connect to Google Drive.
# -------------------------------------------------------------------
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/Avilight_Exports')

if not DATA_DIR.exists():
    raise FileNotFoundError(f'Export folder not found: {DATA_DIR}')

print('Using export folder:', DATA_DIR)


# -------------------------------------------------------------------
# 2. Settings.
# -------------------------------------------------------------------
YEARS = range(2014, 2026)
GRID_KEYS = ['cell_id', 'year', 'month']

N_CELLS = 17835
N_MONTHS = 144
EXPECTED_TOTAL_ROWS = N_CELLS * N_MONTHS  # 2,568,240 rows

MISSING_VALUE = -9999


# -------------------------------------------------------------------
# 3. Confirm the 48 required final grid files are available.
# -------------------------------------------------------------------
required_files = []

for year in YEARS:
    required_files.extend([
        DATA_DIR / f'grid_ndvi_v2_{year}.csv',
        DATA_DIR / f'grid_alan_{year}.csv',
        DATA_DIR / f'grid_lst_day_v2_{year}.csv',
        DATA_DIR / f'grid_rain_{year}.csv'
    ])

missing_files = [file.name for file in required_files if not file.exists()]

if missing_files:
    raise FileNotFoundError(
        'Missing required files:\n' + '\n'.join(missing_files)
    )

print(f'✅ Found all {len(required_files)} final monthly grid CSV files.')


# -------------------------------------------------------------------
# 4. Load and clean one variable across all 12 yearly files.
# -------------------------------------------------------------------
def load_grid_variable(prefix, value_col, valid_col, include_water=False):
    frames = []

    for year in YEARS:
        path = DATA_DIR / f'{prefix}_{year}.csv'

        needed_columns = GRID_KEYS + [value_col, valid_col]

        if include_water:
            needed_columns += ['water_frac', 'is_water']

        available_columns = pd.read_csv(path, nrows=0).columns.tolist()

        absent_columns = [
            column for column in needed_columns
            if column not in available_columns
        ]

        if absent_columns:
            raise ValueError(
                f'{path.name} is missing columns: {absent_columns}\n'
                f'Available: {available_columns}'
            )

        frame = pd.read_csv(
            path,
            usecols=needed_columns,
            dtype={
                'cell_id': 'string',
                'year': 'int16',
                'month': 'int8'
            }
        )

        frames.append(frame)

    df = pd.concat(frames, ignore_index=True)

    if len(df) != EXPECTED_TOTAL_ROWS:
        raise ValueError(
            f'{prefix}: expected {EXPECTED_TOTAL_ROWS:,} rows, '
            f'found {len(df):,}'
        )

    duplicate_rows = df.duplicated(GRID_KEYS).sum()

    if duplicate_rows:
        raise ValueError(
            f'{prefix}: found {duplicate_rows:,} duplicate cell-month rows'
        )

    df[value_col] = pd.to_numeric(
        df[value_col],
        errors='coerce'
    ).astype('float32')

    df[valid_col] = pd.to_numeric(
        df[valid_col],
        errors='coerce'
    ).astype('float32')

    # Convert temporary -9999 placeholders and <30% valid-pixel rows to missing.
    df.loc[
        (df[value_col] <= MISSING_VALUE) |
        (df[valid_col] < 0.3),
        value_col
    ] = np.nan

    if include_water:
        df['water_frac'] = pd.to_numeric(
            df['water_frac'],
            errors='coerce'
        ).astype('float32')

        df['is_water'] = pd.to_numeric(
            df['is_water'],
            errors='coerce'
        ).fillna(0).astype('int8')

    return df


# -------------------------------------------------------------------
# 5. Load all four final monthly grid covariates.
# -------------------------------------------------------------------
grid_ndvi = load_grid_variable(
    prefix='grid_ndvi_v2',
    value_col='ndvi',
    valid_col='ndvi_valid',
    include_water=True
)

grid_alan = load_grid_variable(
    prefix='grid_alan',
    value_col='alan',
    valid_col='alan_valid'
)

grid_lst = load_grid_variable(
    prefix='grid_lst_day_v2',
    value_col='lst_day',
    valid_col='lst_valid'
)

grid_rain = load_grid_variable(
    prefix='grid_rain',
    value_col='rain_mm',
    valid_col='rain_valid'
)

print('✅ Loaded and cleaned NDVI, ALAN, LST, and rain.')


# -------------------------------------------------------------------
# 6. Merge all covariates into one cell × month table.
# -------------------------------------------------------------------
grid_monthly = (
    grid_ndvi
    .merge(
        grid_alan[GRID_KEYS + ['alan', 'alan_valid']],
        on=GRID_KEYS,
        how='outer',
        validate='one_to_one'
    )
    .merge(
        grid_lst[GRID_KEYS + ['lst_day', 'lst_valid']],
        on=GRID_KEYS,
        how='outer',
        validate='one_to_one'
    )
    .merge(
        grid_rain[GRID_KEYS + ['rain_mm', 'rain_valid']],
        on=GRID_KEYS,
        how='outer',
        validate='one_to_one'
    )
    .sort_values(['cell_id', 'year', 'month'])
    .reset_index(drop=True)
)

if len(grid_monthly) != EXPECTED_TOTAL_ROWS:
    raise ValueError(
        f'Final table has {len(grid_monthly):,} rows; '
        f'expected {EXPECTED_TOTAL_ROWS:,}'
    )

print('\nFinal rows:', len(grid_monthly))
print('Expected rows:', EXPECTED_TOTAL_ROWS)

print('\nMissing values after quality filtering:')
print(
    grid_monthly[
        ['alan', 'ndvi', 'lst_day', 'rain_mm']
    ].isna().sum()
)


# -------------------------------------------------------------------
# 7. Save the final monthly grid table for Member 3.
# -------------------------------------------------------------------
output_path = DATA_DIR / 'grid_monthly.parquet'

grid_monthly.to_parquet(
    output_path,
    index=False
)

print(f'\n✅ Saved: {output_path.name}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using export folder: /content/drive/MyDrive/Avilight_Exports
✅ Found all 48 final monthly grid CSV files.
✅ Loaded and cleaned NDVI, ALAN, LST, and rain.

Final rows: 2568240
Expected rows: 2568240

Missing values after quality filtering:
alan        23523
ndvi       975132
lst_day    618880
rain_mm         0
dtype: int64

✅ Saved: grid_monthly.parquet


In [13]:
annual_columns = {
    'alan': 'mean',
    'ndvi': 'mean',
    'lst_day': 'mean',
    'rain_mm': 'mean',
    'alan_valid': 'mean',
    'ndvi_valid': 'mean',
    'lst_valid': 'mean',
    'rain_valid': 'mean',
    'water_frac': 'first',
    'is_water': 'first'
}

grid_annual = (
    grid_monthly
    .groupby(['cell_id', 'year'], as_index=False)
    .agg(annual_columns)
    .sort_values(['cell_id', 'year'])
    .reset_index(drop=True)
)

expected_annual_rows = 17835 * 12

print('Annual rows:', len(grid_annual))
print('Expected rows:', expected_annual_rows)

assert len(grid_annual) == expected_annual_rows

grid_annual.to_parquet(
    DATA_DIR / 'grid_annual.parquet',
    index=False
)

print('✅ Saved grid_annual.parquet')

Annual rows: 214020
Expected rows: 214020
✅ Saved grid_annual.parquet


In [14]:
import pandas as pd

# -------------------------------------------------------------------
# 1. Expected columns.
# -------------------------------------------------------------------
lc_share_columns = [
    'urban_frac',
    'forest_frac',
    'cropland_frac',
    'grass_shrub_frac',
    'wetland_frac',
    'water_lc_frac'
]

province_lc_columns = [
    'adm1',
    'province',
    'adm2_code',
    'year',
    'lc_year_used'
] + lc_share_columns

grid_lc_columns = [
    'cell_id',
    'year',
    'lc_year_used',
    'water_frac',
    'is_water'
] + lc_share_columns


# -------------------------------------------------------------------
# 2. Load only useful columns; discard Earth Engine metadata.
# -------------------------------------------------------------------
province_lc_path = DATA_DIR / 'province_landcover_2014_2025.csv'
grid_lc_path = DATA_DIR / 'grid_landcover_2014_2025.csv'

province_landcover = pd.read_csv(
    province_lc_path,
    usecols=province_lc_columns
)

grid_landcover = pd.read_csv(
    grid_lc_path,
    usecols=grid_lc_columns
)


# -------------------------------------------------------------------
# 3. Set compact numeric types.
# -------------------------------------------------------------------
for df in [province_landcover, grid_landcover]:
    df['year'] = df['year'].astype('int16')
    df['lc_year_used'] = df['lc_year_used'].astype('int16')

    for column in lc_share_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors='coerce'
        ).astype('float32')

province_landcover['adm2_code'] = province_landcover['adm2_code'].astype('int32')

grid_landcover['water_frac'] = pd.to_numeric(
    grid_landcover['water_frac'],
    errors='coerce'
).astype('float32')

grid_landcover['is_water'] = pd.to_numeric(
    grid_landcover['is_water'],
    errors='coerce'
).fillna(0).astype('int8')


# -------------------------------------------------------------------
# 4. Validate expected row counts and the carried-forward 2025 map.
# -------------------------------------------------------------------
expected_province_rows = 85 * 12
expected_grid_rows = 17835 * 12

print('Province land-cover rows:', len(province_landcover))
print('Expected:', expected_province_rows)

print('\nGrid land-cover rows:', len(grid_landcover))
print('Expected:', expected_grid_rows)

assert len(province_landcover) == expected_province_rows
assert len(grid_landcover) == expected_grid_rows

assert (
    province_landcover.loc[
        province_landcover['year'] == 2025,
        'lc_year_used'
    ] == 2024
).all()

assert (
    grid_landcover.loc[
        grid_landcover['year'] == 2025,
        'lc_year_used'
    ] == 2024
).all()

print('\n✅ All 2025 land-cover rows correctly use the 2024 map.')


# -------------------------------------------------------------------
# 5. Inspect class-share sums.
# Small differences from 1 are expected for ungrouped rare classes.
# -------------------------------------------------------------------
province_landcover['lc_share_sum'] = province_landcover[
    lc_share_columns
].sum(axis=1)

grid_landcover['lc_share_sum'] = grid_landcover[
    lc_share_columns
].sum(axis=1)

print('\nProvince class-share sum summary:')
print(province_landcover['lc_share_sum'].describe())

print('\nGrid class-share sum summary:')
print(grid_landcover['lc_share_sum'].describe())


# -------------------------------------------------------------------
# 6. Save final clean Parquet files.
# -------------------------------------------------------------------
province_landcover.to_parquet(
    DATA_DIR / 'province_landcover.parquet',
    index=False
)

grid_landcover.to_parquet(
    DATA_DIR / 'grid_landcover.parquet',
    index=False
)

print('\n✅ Saved province_landcover.parquet')
print('✅ Saved grid_landcover.parquet')

Province land-cover rows: 1020
Expected: 1020

Grid land-cover rows: 214020
Expected: 214020

✅ All 2025 land-cover rows correctly use the 2024 map.

Province class-share sum summary:
count    1020.000000
mean        0.979771
std         0.039565
min         0.666586
25%         0.978552
50%         0.989391
75%         0.995169
max         1.000000
Name: lc_share_sum, dtype: float64

Grid class-share sum summary:
count    214020.000000
mean          0.746443
std           0.423835
min           0.000000
25%           0.424519
50%           1.000000
75%           1.000000
max           1.000000
Name: lc_share_sum, dtype: float64

✅ Saved province_landcover.parquet
✅ Saved grid_landcover.parquet


In [15]:
land_grid_lc = grid_landcover[
    grid_landcover['is_water'] == 0
]

print('Land-cell class-share sum summary:')
print(land_grid_lc['lc_share_sum'].describe())

print('\nLand cells with class-share sum below 0.9:')
print((land_grid_lc['lc_share_sum'] < 0.9).sum())

Land-cell class-share sum summary:
count    159348.000000
mean          0.988045
std           0.074672
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           1.000000
Name: lc_share_sum, dtype: float64

Land cells with class-share sum below 0.9:
5007


In [20]:
from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/Avilight_Exports')

# Find the province file. Your earlier merge may have saved it in /content.
province_candidates = [
    DATA_DIR / 'province_monthly.parquet',
    Path('/content/province_monthly.parquet')
]

province_path = next(
    (path for path in province_candidates if path.exists()),
    None
)

if province_path is None:
    raise FileNotFoundError(
        'province_monthly.parquet was not found in Avilight_Exports or /content.'
    )

# Load saved outputs.
pm = pd.read_parquet(province_path)
gm = pd.read_parquet(DATA_DIR / 'grid_monthly.parquet')
ga = pd.read_parquet(DATA_DIR / 'grid_annual.parquet')
plc = pd.read_parquet(DATA_DIR / 'province_landcover.parquet')
glc = pd.read_parquet(DATA_DIR / 'grid_landcover.parquet')

print('Loaded province file from:', province_path)


# ---------------------------------------------------------------
# 1. Structural checks.
# ---------------------------------------------------------------
print('\n--- STRUCTURE ---')

print('Province monthly rows:', len(pm), '| Expected:', 85 * 144)
print('Grid monthly rows:', len(gm), '| Expected:', 17835 * 144)
print('Grid annual rows:', len(ga), '| Expected:', 17835 * 12)
print('Province land-cover rows:', len(plc), '| Expected:', 85 * 12)
print('Grid land-cover rows:', len(glc), '| Expected:', 17835 * 12)

print('\nDuplicate province-month rows:', pm.duplicated(
    ['adm2_code', 'year', 'month']
).sum())

print('Duplicate grid-month rows:', gm.duplicated(
    ['cell_id', 'year', 'month']
).sum())

print('Duplicate grid-year rows:', ga.duplicated(
    ['cell_id', 'year']
).sum())


# ---------------------------------------------------------------
# 2. Missing-value report.
# ---------------------------------------------------------------
print('\n--- MISSING VALUES (%) ---')

print('Province:')
print(
    pm[['alan', 'ndvi', 'lst_day', 'rain_mm']]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

print('\nGrid:')
print(
    gm[['alan', 'ndvi', 'lst_day', 'rain_mm']]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)


# ---------------------------------------------------------------
# 3. Physical ranges, using land cells only.
# ---------------------------------------------------------------
print('\n--- LAND-CELL VALUE RANGES ---')

land_grid = gm[gm['is_water'] == 0]

print(
    land_grid[
        ['alan', 'ndvi', 'lst_day', 'rain_mm']
    ]
    .describe()
    .loc[['min', 'mean', '50%', 'max']]
    .round(3)
)


# ---------------------------------------------------------------
# 4. Spatial and seasonal checks.
# ---------------------------------------------------------------
print('\n--- SPATIAL CHECKS ---')

grid_cells = gm[['cell_id', 'water_frac', 'is_water']].drop_duplicates()

print('Unique grid cells:', len(grid_cells))
print('Water-heavy cells:', int(grid_cells['is_water'].sum()))

print('\nUP Diliman cell:')
print(
    grid_cells.loc[
        grid_cells['cell_id'] == '072_062'
    ].to_string(index=False)
)

print('\n--- RAINFALL SEASONALITY ---')

monthly_rain = (
    pm.groupby('month', as_index=False)['rain_mm']
    .mean()
    .round(1)
)

print(monthly_rain.to_string(index=False))
print(
    'Four wettest average months:',
    monthly_rain.nlargest(4, 'rain_mm')['month'].tolist()
)


# ---------------------------------------------------------------
# 5. Land-cover and final-file checks.
# ---------------------------------------------------------------
print('\n--- LAND COVER / FINAL FILES ---')

print(
    'Province 2025 uses 2024 land cover:',
    (plc.loc[plc['year'] == 2025, 'lc_year_used'] == 2024).all()
)

print(
    'Grid 2025 uses 2024 land cover:',
    (glc.loc[glc['year'] == 2025, 'lc_year_used'] == 2024).all()
)

for filename in ['provinces.geojson', 'grid.geojson']:
    path = DATA_DIR / filename
    print(
        f'{filename}:',
        f'found ({path.stat().st_size / 1024**2:.2f} MB)'
        if path.exists()
        else 'MISSING'
    )

print('\nFinal Parquet availability:')
for path in [
    province_path,
    DATA_DIR / 'grid_monthly.parquet',
    DATA_DIR / 'grid_annual.parquet',
    DATA_DIR / 'province_landcover.parquet',
    DATA_DIR / 'grid_landcover.parquet'
]:
    print('-', path.name, ':', 'found' if path.exists() else 'MISSING')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded province file from: /content/drive/MyDrive/Avilight_Exports/province_monthly.parquet

--- STRUCTURE ---
Province monthly rows: 12240 | Expected: 12240
Grid monthly rows: 2568240 | Expected: 2568240
Grid annual rows: 214020 | Expected: 214020
Province land-cover rows: 1020 | Expected: 1020
Grid land-cover rows: 214020 | Expected: 214020

Duplicate province-month rows: 0
Duplicate grid-month rows: 0
Duplicate grid-year rows: 0

--- MISSING VALUES (%) ---
Province:
alan       0.08
ndvi       2.61
lst_day    0.29
rain_mm    0.00
dtype: float64

Grid:
alan        0.92
ndvi       37.97
lst_day    24.10
rain_mm     0.00
dtype: float64

--- LAND-CELL VALUE RANGES ---
          alan   ndvi  lst_day   rain_mm
min      0.000 -0.193   -5.910     0.000
mean     3.388  0.719   29.396   220.876
50%      0.629  0.767   28.775   199.200
max   3062.438  0.993   47.946  

In [21]:
lst_outliers = land_grid[
    (land_grid['lst_day'] < 15) |
    (land_grid['lst_day'] > 50)
]

print('LST outlier rows:', len(lst_outliers))
print('Percent of land-cell months:', round(
    len(lst_outliers) / len(land_grid) * 100,
    3
))

print('\nSample outliers:')
print(
    lst_outliers[
        ['cell_id', 'year', 'month', 'lst_day', 'lst_valid']
    ]
    .sort_values('lst_day')
    .head(20)
    .to_string(index=False)
)

LST outlier rows: 1112
Percent of land-cell months: 0.058

Sample outliers:
cell_id  year  month   lst_day  lst_valid
037_038  2016      9 -5.910000   0.507874
038_037  2016      9 -3.590160   0.490196
054_004  2014      7 -3.090000   0.423529
053_004  2014      7 -3.071224   0.576471
073_118  2014     10 -1.970000   0.866667
074_118  2014     10 -1.970000   0.301961
072_118  2014     10 -0.960588   1.000000
090_093  2024      7 -0.929121   0.356863
009_069  2015      8 -0.670000   0.429134
054_005  2014      7 -0.086909   0.429688
053_005  2014      7 -0.028095   0.576471
089_093  2024      7  0.189373   1.000000
011_043  2025      9  0.230000   1.000000
011_044  2025      9  0.754314   1.000000
008_100  2022      9  0.930000   0.490196
072_119  2014     10  1.608824   1.000000
089_092  2024      7  1.614706   0.466667
072_120  2014     10  1.908541   0.913725
083_117  2014     10  2.030000   0.631373
083_116  2014     10  2.030000   0.631373
